In [ ]:
import xarray as xr
from workflow.scripts.utils import read_list_input_paths, make_consistent 
import cartopy.crs as ccrs
from workflow.scripts.plotting_tools import create_facet_plot
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
exp_ustar = read_list_input_paths(snakemake.input.exp_ustar, models_pos=-3)
ctrl_ustar = read_list_input_paths(snakemake.input.ctrl_ustar, models_pos=-3)

if exp_ustar[0].get('EC-Earth3-AerChem'):
    ecm = ctrl_ustar[0]['EC-Earth3-AerChem']
    exp_ustar[0]['EC-Earth3-AerChem'] = exp_ustar[0]['EC-Earth3-AerChem'].reindex({"lon": ecm.lon, "lat": ecm.lat}, method="nearest")

In [ ]:
diff = {m : exp_ustar[0][m].isel(time=slice(2,None)).mean(dim='time') - ctrl_ustar[0][m].isel(time=slice(2,None)).mean(dim='time') for m in exp_ustar[0]}

In [ ]:
models = sorted(diff.keys())

In [ ]:
fig,ax,cax = create_facet_plot(len(models), subplot_kw={'projection':ccrs.EckertIV()})
for ai in ax.keys():
    ax[ai].coastlines()
norm = mpl.colors.Normalize(vmin=-.04, vmax=0.04)
cmap = mpl.cm.PRGn.resampled(12)
for m, ak in zip(models, ax.keys()):
    diff[m]['ustar'].plot(ax=ax[ak], add_colorbar=False, transform=ccrs.PlateCarree(), norm=norm, cmap=cmap)
    ax[ak].set_title(m)
fig.colorbar(mpl.cm.ScalarMappable(norm,cmap),cax=cax)
cax.set_ylabel('Friction velocy change [m s-1]')
plt.subplots_adjust(wspace=.035, hspace=.02)
plt.savefig(snakemake.output.outpath, dpi=300, bbox_inches='tight')